In [1]:
# Get raw data
with open('input/24.txt', 'r') as f:
    rawinput = f.read().strip()

In [2]:
import numpy as np

In [3]:
# Part 1
pos, vel = [np.array(i) 
            for i in zip(*[[[int(l) for l in k.split(',')] for k in j] 
                           for i in rawinput.split('\n') 
                           for j in [i.split('@')]])]
m = vel[:,1]/vel[:,0]
c = pos[:,1]-(pos[:,0]*m)

pair_ix = np.array([[i,j]
                    for i in range(pos.shape[0]-1)
                    for j in range(i+1,pos.shape[0])])

x_int = np.where((z:=np.sum(m[pair_ix]*[[1,-1]], axis=1))==0, 
                 np.nan,
                 np.sum(c[pair_ix]*[[-1,1]], axis=1) / np.where(z==0,1,z))
y_int = x_int * m[pair_ix][:,0] + c[pair_ix][:,0]

future_int = ((x_int >= 200000000000000) & (x_int <= 400000000000000)
              & (y_int >= 200000000000000) & (y_int <= 400000000000000)
              & np.all((np.expand_dims(x_int,1)-pos[pair_ix][:,:,0])
                       /vel[pair_ix][:,:,0] >= 0, 
                       axis=1))

np.sum(future_int).item()

12343

In [ ]:
# Part 2
# (Adding comments because I'll never remember how this works)
# Get starting position and position after N nanoseconds for 3 hailstones
ht = 1000000000000
pos3, vel3 = pos[:3], vel[:3]
pos3_by_t = np.expand_dims(pos3,1) + np.expand_dims(vel3,1) * [[[0], [ht]]]

# Calculate positions for HS1&2 relative to HS0
# Divide each by its own z component, so parallel vectors are represented by the same x and y value pair
# Map linear relationship between normalized x and y for each HS1&2, then identify normalized x value where lines intersect
vecs = (z:=pos3_by_t[1:]-pos3_by_t[:1])[:,:,:-1] / z[:,:,-1:]
sl = (z:=vecs[:,1]-vecs[:,0])[:,1]/z[:,0]
ic = vecs[:,0,1] - sl*vecs[:,0,0]
int_x = ((ic[1]-ic[0])/(sl[0]-sl[1])).item()

# Run a binary search to identify the corresponding time for each of HS1&2
get_test_ix = lambda i,t: ((z:=pos3[i]-pos3[0] + t*(vel3[i]-vel3[0]))[0]/z[-1]).item()
time_ix = []
for i in [1,2]:
    if (vel3[i,2]-vel3[0,2]) != 0 and (zpt:=(pos3[0,2]-pos3[i,2]) / (vel3[i,2]-vel3[0,2]))>0:
        if get_test_ix(i,zpt+1)*int_x > 0:
            test_t = [int(k) for k in [zpt+1, 10**np.ceil(np.log10(zpt+1))]]
        else:
            test_t = [0, int(zpt)]
    else:
        test_t = [0,1]
    test_vs = [get_test_ix(i,t)-int_x for t in test_t]
    while np.prod(test_vs[-2:]) > 0:
        test_t += [test_t[-1]*10]
        test_vs += [get_test_ix(i,test_t[-1])-int_x]

    test_lims = test_t[-2:]
    while test_lims[1]-test_lims[0]>1:
        test_val = sum(test_lims)//2
        test_lims[(get_test_ix(i,test_val)-int_x)*test_vs[-1] > 0] = test_val
        
    time_ix += [test_lims[0]]
    
# Find the absolute positions of HS1&2 at these times, and calculate formula for x,y and z coords of rock in terms of t
rock_sl = (((pos[2]+time_ix[1]*vel[2])-(pos[1]+time_ix[0]*vel[1])) / (time_ix[1]-time_ix[0])).astype(int)
rock_ic = pos[1] + time_ix[0] * (vel[1] - rock_sl)

# Intercept term == starting point for rock, so sum of terms is final answer
sum(rock_ic).item()

769281292688187